<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/HEADSWAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
#@title 1. Install dependencies
!pip install -q --upgrade "torchao>=0.16.0" "diffusers>=0.33.0" transformers accelerate peft safetensors pillow opencv-python-headless fastapi uvicorn python-multipart huggingface_hub

from IPython.display import clear_output
clear_output()
print("✅ Installed")

✅ Installed


In [60]:
#@title 2. Download & verify Qwen-Image 2.1 + BFS Head Swap LoRA weights
import gc
import torch
from huggingface_hub import hf_hub_download, snapshot_download

torch.cuda.empty_cache()
gc.collect()

print("Downloading/verifying base model & LoRA weights...")
snapshot_download(
    repo_id="Qwen/Qwen-Image-2.1",
    ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
)
lora_path = hf_hub_download(
    repo_id="Alissonerdx/BFS-Best-Face-Swap",
    filename="bfs_head_v1.1_qwen_2.1.safetensors",
)
print(f"✅ Weights cached. LoRA path: {lora_path}")

✅ Notebook kernel GPU memory cleared.


In [82]:
#@title 3A. Clear kernel VRAM
import gc
import torch

if "pipe" in globals():
    del pipe

gc.collect()
torch.cuda.empty_cache()
print("✅ Notebook kernel GPU memory cleared.")

✅ Notebook kernel GPU memory cleared.


In [83]:
%%writefile /content/headswap_server.py
import base64
import gc
import os
import sys
import traceback
from io import BytesIO
from typing import Optional

import torch
from diffusers import DiffusionPipeline
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from huggingface_hub import hf_hub_download
from PIL import Image
from pydantic import BaseModel
from safetensors.torch import load_file

try:
    from torchao.quantization import int8_weight_only, quantize_
    HAS_TORCHAO = True
except ImportError:
    HAS_TORCHAO = False

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

pipe: Optional[DiffusionPipeline] = None
MAX_SIDE = 1024


class HeadSwapRequest(BaseModel):
    body: str
    head: str
    prompt: str = ""
    num_inference_steps: int = 12
    guidance_scale: float = 1.0


def dataurl_to_pil(dataurl: str) -> Image.Image:
    if "," in dataurl:
        dataurl = dataurl.split(",", 1)[1]
    image_data = base64.b64decode(dataurl)
    return Image.open(BytesIO(image_data)).convert("RGB")


def pil_to_dataurl(img: Image.Image) -> str:
    buffered = BytesIO()
    if img.mode != "RGB":
        img = img.convert("RGB")
    img.save(buffered, format="JPEG", quality=95)
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{img_str}"


def resize_max_side(img: Image.Image, max_side: int = MAX_SIDE) -> Image.Image:
    w, h = img.size
    largest = max(w, h)
    if largest <= max_side:
        new_w = max(16, (w // 16) * 16)
        new_h = max(16, (h // 16) * 16)
        if (new_w, new_h) != (w, h):
            return img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        return img
    scale = max_side / float(largest)
    new_w = max(16, (int(round(w * scale)) // 16) * 16)
    new_h = max(16, (int(round(h * scale)) // 16) * 16)
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)


def map_bfs_lora_state_dict(state_dict: dict) -> dict:
    """Normalizes ComfyUI/BFS LoRA keys if img_mlp.gate_up keys appear."""
    mapped = {}
    for k, v in state_dict.items():
        nk = k
        if nk.startswith("diffusion_model."):
            nk = nk[len("diffusion_model."):]
        if not nk.startswith("transformer.") and "transformer_blocks." in nk:
            nk = "transformer." + nk
        mapped[nk] = v
    return mapped


@app.on_event("startup")
def load_model():
    global pipe
    print("Loading Qwen-Image 2.1 + BFS Head Swap LoRA with optimizations...", flush=True)
    torch.cuda.empty_cache()
    gc.collect()

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    pipe = DiffusionPipeline.from_pretrained(
        "Qwen/Qwen-Image-2.1",
        torch_dtype=dtype,
    )

    lora_file = hf_hub_download(
        repo_id="Alissonerdx/BFS-Best-Face-Swap",
        filename="bfs_head_v1.1_qwen_2.1.safetensors",
    )

    try:
        pipe.load_lora_weights(lora_file)
    except Exception as lora_err:
        print(f"Standard load_lora_weights fallback triggered: {lora_err}", flush=True)
        raw_sd = load_file(lora_file)
        mapped_sd = map_bfs_lora_state_dict(raw_sd)
        pipe.load_lora_weights(mapped_sd)

    if HAS_TORCHAO and hasattr(pipe, "transformer") and pipe.transformer is not None:
        print("Applying torchao int8_weight_only quantization to transformer...", flush=True)
        try:
            quantize_(pipe.transformer, int8_weight_only())
            print("Quantization applied successfully.", flush=True)
        except Exception as q_err:
            print(f"Quantization warning: {q_err}", flush=True)

    pipe.enable_model_cpu_offload()
    pipe.enable_attention_slicing()
    if hasattr(pipe, "vae") and pipe.vae is not None:
        if hasattr(pipe.vae, "enable_tiling"):
            pipe.vae.enable_tiling()
        if hasattr(pipe.vae, "enable_slicing"):
            pipe.vae.enable_slicing()

    torch.cuda.empty_cache()
    gc.collect()
    print("Model + LoRA loaded and optimized", flush=True)


@app.get("/health")
def health():
    return {"status": "ok", "loaded": pipe is not None}


@app.post("/headswap")
def headswap(req: HeadSwapRequest):
    try:
        if pipe is None:
            return {"error": "Pipeline not initialized"}

        raw_body = dataurl_to_pil(req.body)
        raw_head = dataurl_to_pil(req.head)

        body_img = resize_max_side(raw_body, MAX_SIDE)
        head_img = resize_max_side(raw_head, MAX_SIDE)

        prompt = req.prompt.strip() or (
            "head_swap: start with the first image as the base image, keeping its lighting, "
            "environment, and background. remove the head from the first image completely and "
            "replace it with the head from the second image, strictly preserving the hair, eye "
            "color, nose structure from the second image. copy the direction of the eye, head "
            "rotation, micro expressions from the first image, high quality, sharp details, 4k"
        )

        print(
            f"Running head swap... raw_body={raw_body.size} -> body={body_img.size}, "
            f"raw_head={raw_head.size} -> head={head_img.size}",
            flush=True,
        )

        torch.cuda.empty_cache()
        gc.collect()

        call_kwargs = {
            "prompt": prompt,
            "image": [body_img, head_img],
            "num_inference_steps": req.num_inference_steps,
            "height": body_img.height,
            "width": body_img.width,
        }
        if "guidance_scale" in pipe.__call__.__code__.co_varnames:
            call_kwargs["guidance_scale"] = req.guidance_scale

        with torch.inference_mode():
            result = pipe(**call_kwargs).images[0]

        if result.size != raw_body.size:
            result = result.resize(raw_body.size, Image.Resampling.LANCZOS)

        print("Done.", flush=True)
        return {"image": pil_to_dataurl(result)}

    except Exception as e:
        error_msg = traceback.format_exc()
        print(error_msg, flush=True)
        return {"error": str(e)}
    finally:
        torch.cuda.empty_cache()
        gc.collect()


if __name__ == "__main__":
    import uvicorn
    print("Server listening...", flush=True)
    uvicorn.run(app, host="127.0.0.1", port=8000)

Overwriting /content/headswap_server.py


In [84]:
#@title 4. Start server + Cloudflare tunnel
import os
import re
import subprocess
import sys
import time

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart"],
    check=True,
)

SERVER_SCRIPT = "/content/headswap_server.py"
SERVER_LOG = "/content/headswap_server.log"
PORT = 8000
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Kill any previous server / tunnel
subprocess.run(["pkill", "-f", "headswap_server.py"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Download cloudflared if needed
if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            CLOUDFLARED,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

# Start the FastAPI server
print("Starting Head Swap server...")
with open(SERVER_LOG, "w") as log:
    server_proc = subprocess.Popen(
        [sys.executable, SERVER_SCRIPT],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

# Wait until the model is loaded
print("Waiting for model to load (this can take 1–2 minutes)...")
model_ready = False
for _ in range(180):
    time.sleep(2)
    if os.path.exists(SERVER_LOG):
        with open(SERVER_LOG) as f:
            content = f.read()
            if "Model + LoRA loaded" in content or "Server listening" in content:
                print("✅ Model loaded")
                model_ready = True
                break

if not model_ready:
    print("⚠️ Model took too long. Check the log below.")
    !tail -30 /content/headswap_server.log

# Start Cloudflare tunnel
print("Starting Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

TUNNEL_URL = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if m:
        TUNNEL_URL = m.group(0)
        break

if TUNNEL_URL:
    print("\n" + "=" * 60)
    print(f"READY: {TUNNEL_URL}")
    print("=" * 60)
    print("\n→ Copy the URL above and paste it into the Head Swap tab on your website.")
else:
    print("❌ Could not get Cloudflare URL. Try re-running this cell.")

Starting Head Swap server...
Waiting for model to load (this can take 1–2 minutes)...
✅ Model loaded
Starting Cloudflare tunnel...

READY: https://wool-flower-describes-filing.trycloudflare.com

→ Copy the URL above and paste it into the Head Swap tab on your website.


In [74]:
!tail -50 /content/headswap_server.log

W1115 21:02:39.961000 16377 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 21:02:40.008000 16377 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
/content/headswap_server.py:78: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
Server listening...
INFO:     Started server process 